# Evidencias historicas: empresas retiradas da bolsa
Execute as quatro celulas em ordem. Este notebook cria uma pasta nova: nao altera seu notebook de analise nem apaga coletas anteriores.

A chave Tiingo fica nos Segredos do Colab (`TIINGO_API_KEY`) ou em uma entrada oculta. Nunca cole a chave em uma celula ou mensagem. Nao e necessario instalar dependencias.

O ZIP contem precos, respostas completas do provedor, hashes e uma copia do codigo. Mantenha-o privado e respeite o licenciamento. Cobertura tecnica aprovada **nao** significa reconciliacao economica aprovada: o pacote nao libera benchmark nem recalibracao.

In [ ]:
# 1. Preparar uma copia nova do codigo e registrar a versao usada.
from pathlib import Path
import hashlib
import json
import subprocess
import sys
import tempfile
import zipfile

session = Path(tempfile.mkdtemp(prefix='lifecycle_evidence_', dir='/content'))
repo = session / 'repository'
bundle = session / 'lifecycle_evidence'
bundle.mkdir()
evidence = bundle / 'evidence'
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'master',
                'https://github.com/fred-ykv/Value-Investing-In-Python.git', str(repo)], check=True)
commit = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
if not (repo / 'collect_lifecycle_evidence.py').is_file():
    raise RuntimeError('O coletor ainda nao esta nesta versao do master.')
subprocess.run(['git', '-C', str(repo), 'archive', '--format=zip',
                '--output=' + str(bundle / 'source.zip'), commit], check=True)
(bundle / 'source_commit.txt').write_text(commit + '\n', encoding='ascii')
print('Codigo preparado. Versao:', commit)
print('Pasta nova:', session)

In [ ]:
# 2. Coletar dez empresas. A chave nao entra nos arquivos ou argumentos.
def run_private_collection():
    import os
    from getpass import getpass
    from google.colab import userdata
    try:
        token = userdata.get('TIINGO_API_KEY')
    except Exception:
        token = None
    if not token:
        token = getpass('Chave Tiingo (entrada oculta): ')
    token = (token or '').strip()
    if not token:
        raise ValueError('Chave vazia. Configure o segredo TIINGO_API_KEY e execute esta celula novamente.')
    child_env = dict(os.environ, TIINGO_API_KEY=token)
    try:
        completed = subprocess.run([sys.executable, str(repo / 'collect_lifecycle_evidence.py'),
                                    '--outdir', str(evidence)], cwd=repo, env=child_env,
                                   text=True, capture_output=True, check=False)
        print(completed.stdout.replace(token, '[redacted]'))
        if completed.returncode:
            print(completed.stderr.replace(token, '[redacted]'))
            print('Coleta incompleta. Se houver manifesto, as proximas celulas exportam o diagnostico.')
    finally:
        child_env.pop('TIINGO_API_KEY', None)
        del token
    return completed.returncode

collection_exit_code = run_private_collection()

In [ ]:
# 3. Verificar integridade antes de empacotar, mesmo em coletas parciais.
from IPython.display import Markdown, display

def verify_evidence():
    manifest_path = evidence / 'manifest.json'
    if not manifest_path.is_file():
        raise RuntimeError('Coleta sem manifesto. Confira a celula 2; para nova tentativa, execute desde a celula 1.')
    digest = lambda data: hashlib.sha256(data).hexdigest()
    manifest_bytes = manifest_path.read_bytes()
    if digest(manifest_bytes) != (evidence / 'manifest.sha256').read_text().strip():
        raise RuntimeError('Manifesto alterado; pacote nao exportado.')
    report = json.loads(manifest_bytes)
    for name, expected in report['files_sha256'].items():
        if digest((evidence / name).read_bytes()) != expected:
            raise RuntimeError('Arquivo alterado: ' + name)
    for response in report['responses']:
        if response['status'] == 'captured':
            data = (evidence / response['path']).read_bytes()
            if digest(data) != response['sha256'] or len(data) != response['bytes']:
                raise RuntimeError('Resposta alterada: ' + response['path'])
    with zipfile.ZipFile(bundle / 'source.zip') as source:
        for name, expected in report['code_sha256'].items():
            if digest(source.read(name).replace(b'\r\n', b'\n')) != expected:
                raise RuntimeError('Codigo coletado difere da copia arquivada: ' + name)
    return report

report = verify_evidence()
display(Markdown((evidence / 'coverage_report.md').read_text(encoding='utf-8')))
print('Integridade dos arquivos: aprovada. Reconciliacao economica: pendente.')

In [ ]:
# 4. Baixar o pacote privado, com o codigo e a evidencia completa.
from google.colab import files

report = verify_evidence()
package_metadata = {'source_commit': commit, 'collection_exit_code': collection_exit_code,
                    'source_zip_sha256': hashlib.sha256((bundle / 'source.zip').read_bytes()).hexdigest(),
                    'evidence_manifest_sha256': hashlib.sha256((evidence / 'manifest.json').read_bytes()).hexdigest(),
                    'capture_complete': report['capture_complete'], 'eligible_for_benchmark': False}
(bundle / 'package.json').write_text(json.dumps(package_metadata, indent=2), encoding='utf-8')
archive = session / 'lifecycle_evidence.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as output:
    for path in sorted(bundle.rglob('*')):
        if path.is_file():
            output.write(path, path.relative_to(session).as_posix())
print('Pacote:', archive.name)
print('Coleta:', 'completa' if report['capture_complete'] else 'incompleta; enviar para diagnostico')
print('Guarde este ZIP de forma privada. Ele ainda nao e um benchmark aprovado.')
files.download(str(archive))